In [0]:
-- How to read DESCRIBE HISTORY (one row = one write to the table)

-- Full load                                   = DLT REFRESH + STREAMING UPDATE (all rows)
-- Incremental processing                      = STREAMING UPDATE (new file rows only; nothing if no new file)
-- Finance group + incremental (finance_daily) = STREAMING UPDATE (new file rows only; nothing if no new file)
-- NOT finance group (finance_daily run)       = nothing: no DLT REFRESH, no STREAMING UPDATE
-- Note: to tell "incremental" from "skipped", a new file must land first

-- Test 1: run bronze_layer with refresh_group = all
-- Full refresh table: expect a new DLT REFRESH, then STREAMING UPDATE with numOutputRows = all rows (79)

DESCRIBE HISTORY adb_alejandro_research_dev_aue_ws.raw_delta.bronze_dim_investment_option LIMIT 5;

-- Normal refresh table: expect no DLT REFRESH; a new STREAMING UPDATE only if a new file I donlanded
DESCRIBE HISTORY adb_alejandro_research_dev_aue_ws.raw_delta.bronze_dim_fund LIMIT 5;

-- Test 2: run bronze_layer with refresh_group = finance_daily only
-- This table is not part of the finance group and should be ignored (no new DLT REFRESH)
DESCRIBE HISTORY adb_alejandro_research_dev_aue_ws.raw_delta.bronze_dim_investment_option LIMIT 2;

-- This table is part of the finance group and should get a normal refresh
-- (no DLT REFRESH; a new STREAMING UPDATE only with the rows of a new file)
DESCRIBE HISTORY adb_alejandro_research_dev_aue_ws.raw_delta.bronze_fact_investment_strategic_allocation LIMIT 3;

--------------------------------------------------------------------------------------------------------------
-- =====================================================================================
-- TEST 2: refresh_group = finance_daily (selective refresh)
-- Goal : prove that a finance run loads ONLY tables tagged refresh.group = finance_daily,
--        and skips every other table, even if those have new files.
--
-- Setup:
--   1. Upload a NEW csv (new file name, never overwrite) for:
--        - fact_investment_strategic_allocation  (finance_daily, normal refresh)
--        - dim_fund                              (no group, normal refresh)
--   2. Job etas_process_apra > Run now with different settings
--        > job parameter refresh_group = finance_daily > run task bronze_layer only.
--
-- How to read DESCRIBE HISTORY:
--   DLT REFRESH + STREAMING UPDATE = full refresh (table emptied, all files re-read)
--   STREAMING UPDATE only          = normal (incremental) refresh, numOutputRows = new rows
--   No new version                 = table skipped in this run
--
-- Result 17 Nov 2026: PASSED (all 3 checks below)
-- =====================================================================================

-- Check 1: finance table, normal refresh -> loaded
-- Expect: NEW STREAMING UPDATE, numOutputRows = 124130 (rows of the new file), no DLT REFRESH
-- Result: v5 17 Nov 01:27 UTC, 124130 rows, no DLT REFRESH
DESCRIBE HISTORY adb_alejandro_research_dev_aue_ws.raw_delta.bronze_fact_investment_strategic_allocation LIMIT 3;

-- Check 2: not finance, has a new file -> skipped
-- Expect: NOTHING new (the new file waits for the next 'all' run)
-- Result: latest still v2 from 12 Nov 04:41 UTC
DESCRIBE HISTORY adb_alejandro_research_dev_aue_ws.raw_delta.bronze_dim_fund LIMIT 3;

-- Check 3: not finance, full refresh table (refresh.full_override = true) -> skipped
-- Expect: NO new DLT REFRESH (full_override only applies when the table is in the run)
-- Result: no new version in the finance run
DESCRIBE HISTORY adb_alejandro_research_dev_aue_ws.raw_delta.bronze_dim_investment_option LIMIT 3;


